## 1. Data Loading, inspection and preparation 

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
import warnings
import ssl
import pandas as pd

from keras import Sequential
from keras.layers import Input, Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization, RandomFlip, RandomRotation, RandomZoom


Configuration set-up

In [ ]:
warnings.filterwarnings("ignore")
ssl._create_default_https_context = ssl._create_unverified_context

# Select TensorFlow's backend
os.environ["KERAS_BACKEND"] = "tensorflow"

# Select hardware accelerator (GPU if available, otherwise CPU)
#os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
gpu_devices = tf.config.list_physical_devices("GPU")
if gpu_devices:
    details = tf.config.experimental.get_device_details(gpu_devices[0])
    gpu_name = details.get("device_name", "Name not found")
    print(f"Available GPU: {gpu_name}")
else:
    print("No GPU detected.")

print(f"Keras version: {keras.__version__}")

seed = 42
keras.utils.set_random_seed(seed)
np.random.seed(seed)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Import the dataset using tenserflow dataset:

In [ ]:
dataset_split = ["train[:80%]", "train[80%:]", "test"]
(train_split, val_split, test_data), info = tfds.load("oxford_iiit_pet", split=dataset_split, as_supervised=True, with_info=True)

Now we have a "lazy" object that we need to convert in a usable dataset:

In [ ]:
def dataset_to_numpy(dataset, img_size):
    images = []
    labels = []
    
    for img, label in tfds.as_numpy(dataset):
        img_resized = tf.image.resize(img, img_size).numpy().astype(np.uint8)
        images.append(img_resized)
        labels.append(label)

    return np.array(images), np.array(labels)

IMG_SHAPE = (224, 224)

x_train, y_train = dataset_to_numpy(train_split, img_size=IMG_SHAPE)
x_val, y_val     = dataset_to_numpy(val_split, img_size=IMG_SHAPE)
x_test, y_test   = dataset_to_numpy(test_data, img_size=IMG_SHAPE)


Check the class distribution in every split:

In [ ]:
_, train_count = np.unique(y_train, return_counts=True)
_, validation_count = np.unique(y_val, return_counts=True)
_, test_count = np.unique(y_test, return_counts=True)

distribution = pd.DataFrame({
    "Class": info.features["label"].names,
    "Train%": (train_count / len(y_train)) * 100,
    "Validation%": (validation_count / len(y_val)) * 100,
    "Test%": (test_count / len(y_test)) * 100,
    "Train": train_count,
    "Validation": validation_count,
    "Test": test_count,
})
distribution["Total count"] = distribution["Train"] + distribution["Validation"] + distribution["Test"]
distribution = distribution.round(2)

print(f"Perfectly balanced dataset distribution if ~={100/len(distribution):.2f}%:")
distribution

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(20, 15))
ax = ax.flatten()

for idx in range(3):
    ax[idx].bar(distribution["Class"], distribution.iloc[:, idx + 1], color="skyblue")
    ax[idx].set_title(f"{distribution.columns[idx + 1]} Distribution")
    ax[idx].set_xlabel("Class")
    ax[idx].set_ylabel("Percentage (%)")
    ax[idx].set_xticklabels(distribution["Class"], rotation=90)

plt.tight_layout()
plt.show()

Let's get the "labels" for further usage:

In [ ]:
labels = {i: label for i, label in enumerate(info.features["label"].names)}
#labels 

Let's check if the operation done so far are correct:

In [ ]:
n_train, w_train, h_train, channels = x_train.shape
n_test, w_test, h_test, channels_test = x_test.shape
n_classes = len(np.unique(y_train))

print(f"Number of classes: {n_classes}\n")
print(f"There are {n_train} samples in the training set, with size ({w_train},{h_train}).")
print(f"There are {n_test}  samples in the test set with, size    ({w_test},{h_test}).\n")
print(f"There are {n_classes} different classes.")

Let's normalize the data to improve the perfomance:

In [ ]:
x_train = x_train.astype("float") / np.max(x_train)
x_test = x_test.astype("float") / np.max(x_test)
print(f"The new range of the images is [{x_train.min()},{x_train.max()}].")

input_shape = x_train.shape[1:]

## 

## 2. Naive model implementation 

In [ ]:
model = Sequential([
    # Input layer.
        Input(shape = input_shape),

        Conv2D(filters = 32, kernel_size = (3, 3), activation = "relu", padding = 'same'),
        BatchNormalization(),
        MaxPooling2D((2,2)),
        Dropout(0.25),

        Conv2D(filters = 64, kernel_size = (3, 3), activation = "relu", padding = 'same'),
        BatchNormalization(),
        MaxPooling2D((2,2)),
        Dropout(0.25),

        Conv2D(filters = 128, kernel_size = (3, 3), activation = "relu", padding = 'same'),
        BatchNormalization(),
        MaxPooling2D((2,2)),
        Dropout(0.25),

        Flatten(),
        Dense(256, activation="relu"),
        BatchNormalization(),
        Dropout(0.5),
        Dense(units = n_classes, activation = "softmax")
])

model.summary()

In [ ]:
from keras.optimizers import Adam
from keras.losses import SparseCategoricalCrossentropy

opt = Adam(learning_rate = 1e-3)
loss_fcn = SparseCategoricalCrossentropy()

model.compile(loss = loss_fcn,
              optimizer = opt, 
              metrics = ["accuracy"])

batch_size = 128
epochs = 6
val_split_percentage = 0.25

model.fit(x_train, 
          y_train, 
          batch_size = batch_size, 
          epochs = epochs, 
          validation_split = val_split_percentage);

In [ ]:
plt.figure(figsize=(20, 3))
for i, metric in enumerate(["accuracy", "loss"]):
    plt.subplot(1, 2, i + 1) 
    plt.plot(model.history.history[metric])
    plt.plot(model.history.history["val_" + metric])
    plt.title("Model {}".format(metric))
    plt.xlabel("epochs")
    plt.ylabel(metric)
    plt.legend(["train", "val"])

test_loss, test_metric = model.evaluate(x_test, y_test, verbose = 1)
print(f"The test loss is {test_loss:.4f}, the test accuracy is {test_metric:.4f}.")

pred = model.predict(x_test)
print("Test Input Shape: {} Test output shape: {}".format(x_test.shape, pred.shape))


In [ ]:
pred = np.argmax(pred, axis=-1)
y_test = y_test.squeeze()

plt.figure(figsize=(20,12))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    rand_idx = np.random.randint(0, x_test.shape[0])
    plt.axis('off')
    #Qui da errore sul titolo, da mettere a posto
    plt.title(f"Pred : {labels[pred[rand_idx]]} - GT : {labels[y_test[rand_idx]]}")
    plt.imshow(x_test[rand_idx], cmap="gray")
plt.show()